# Records from PhysioNet

PhysioNet publishes its databases at
`https://physionet.org/files/<database>/<version>/`. The simplest way to
use a record is to download its files (header, signal files, annotation
files) and add their directory to the WFDB path.

This notebook downloads record 100 of the MIT-BIH Arrhythmia Database
(`mitdb`, version 1.0.0, about 2 MB). Without a network connection it uses
the sample record `100s` (the first minute of the same record) instead.

In [ ]:
import os
import urllib.request

import numpy as np
import wfdbx

BASE = "https://physionet.org/files/mitdb/1.0.0/"
DEST = "mitdb"


def download(record, extensions, base=BASE, dest=DEST):
    """Download the files of a record (skipping those already present)."""
    os.makedirs(dest, exist_ok=True)
    for ext in extensions:
        path = os.path.join(dest, f"{record}.{ext}")
        if not os.path.exists(path):
            urllib.request.urlretrieve(f"{base}{record}.{ext}", path)


try:
    download("100", ["hea", "dat", "atr"])
    record = "100"
except OSError as err:
    print("Could not download from PhysioNet:", err)
    print("Using the sample record 100s instead.")
    record = "100s"

wfdbx.setwfdb(". " + DEST + " " + wfdbx.sample_data_dir())

In [ ]:
hdr = wfdbx.read_header(record)
rec = wfdbx.read_record(record, sampto="0:10", physical=True)
ann = wfdbx.read_annotations(record, "atr")
print(f"record {record}: {len(hdr)} signals at {rec.fs} Hz, "
      f"{len(ann)} annotations")
print(rec.names, rec.signals[:3])

The same works for any PhysioNet database: change `BASE` to the database's
`files` address, and download the files listed in its `RECORDS` file and in
each record's header.